In [ ]:
from google.colab import drive, userdata

userdata.get('HF_TOKEN')
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile

!unzip /content/drive/MyDrive/data_preprocessed_llava.zip

Archive:  /content/drive/MyDrive/data_preprocessed_llava.zip
   creating: data_preprocessed_llava/
  inflating: data_preprocessed_llava/dataset_dict.json  
   creating: data_preprocessed_llava/train/
  inflating: data_preprocessed_llava/train/data-00000-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00001-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00002-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00003-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00004-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00005-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00006-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00007-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00008-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00009-of-00021.arrow  
  inflating: data_preprocessed_llava/train/data-00010-of-00021.arrow  
  inflating: data_p

# Training

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from datasets import load_from_disk
from transformers import AutoProcessor, LlavaForConditionalGeneration
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
import os

In [ ]:
# -----------------------------
# Config
# -----------------------------
DATAPATH = "./data_preprocessed_llava"
BATCH_SIZE = 8
NUM_EPOCHS = 15
LR = 5e-5
WEIGHT_DECAY = 0.01
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Load dataset
# -----------------------------
print("loading from disk")
data = load_from_disk(DATAPATH)
data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_loader = DataLoader(data["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(data["validation"], batch_size=BATCH_SIZE, num_workers=0)


In [13]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# -----------------------------
# Load LLaVA model + processor
# docs: https://huggingface.co/docs/transformers/en/model_doc/llava
# -----------------------------

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
llava = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
llava.model

LlavaModel(
  (vision_tower): CLIPVisionModel(
    (vision_model): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(577, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGEL

In [6]:
import torch
import torch.nn as nn

class LLaVAClassifier(nn.Module):
    def __init__(self, llava_model, hidden_size=4096):
        super().__init__()
        self.llava = llava_model

        # Add a linear classifier head
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size, eps=1e-5, elementwise_affine=True),
            nn.Linear(hidden_size, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1)   # binary classification
        ).float()

    def forward(self, pixel_values, input_ids, attention_mask):
        # Forward pass through LLaVA
        outputs = self.llava(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True,
            output_hidden_states=True,
        )

        # Use the last token of the last hidden state
        # shape: (batch_size, seq_len, hidden_dim)
        last_hidden = outputs.hidden_states[-1].mean(dim=1).float()
        logits = self.classifier(last_hidden).squeeze(-1)
        return logits


In [ ]:
# -----------------------------
# Freeze all backbone parameters
# -----------------------------
for param in llava.parameters():
    param.requires_grad = False
    param.data = param.data.to("cpu")

# Wrap with classifier
model = LLaVAClassifier(llava, hidden_size=llava.config.text_config.hidden_size).to(device)

# -----------------------------
# Compute class imbalance for pos_weight
# -----------------------------
labels = data["train"]["labels"]
pos_count = sum(labels)
neg_count = len(labels) - pos_count
pos_weight = neg_count / max(pos_count, 1)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device, dtype=torch.float16))
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

/tmp/ipykernel_1644/286557970.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device, dtype=torch.float16))


In [ ]:
# model.train()
# batch = None
# for b in tqdm(train_loader, desc=f"Train Epoch {1}", leave=False):
#     batch = b
#     break

# pixel_values = batch["pixel_values"].to(device)
# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)
# labels = batch["labels"].float().to(device)

# optimizer.zero_grad()

# llava_sample_out = llava(
#     input_ids=input_ids,
#     attention_mask=attention_mask,
#     pixel_values=pixel_values,
#     return_dict=True,
#     output_hidden_states=True,
# )

# print(llava_sample_out.hidden_states[-1].shape)
# print(llava_sample_out.hidden_states[-1][:, -1, :].shape)

torch.Size([8, 1024, 4096])
torch.Size([8, 4096])


In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)

# -----------------------------
# Training loop
# -----------------------------
best_acc = 0
train_losses, val_losses, val_accs = [], [], []

scaler = GradScaler()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Train Epoch {epoch}", leave=False):
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device, dtype=torch.float32)

        optimizer.zero_grad()
        with autocast(dtype=torch.float16):
            logits = model(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} Train Loss: {train_loss}")
    train_losses.append(train_loss)

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()
    val_loss = 0
    correct, total = 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Val Epoch {epoch}", leave=False):
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device, dtype=torch.float32)

            with autocast(dtype=torch.float16):
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    print(f"Epoch {epoch} Validation Loss: {val_loss}")
    print(f"Epoch {epoch} Validation Accuracy: {val_acc}")
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_acc > best_acc:
      best_acc = val_acc
      torch.save(model.state_dict(), f"checkpoints/best_model_epoch_{epoch}.pt")
      print("Saved best model!")

torch.save(model.state_dict(), f"checkpoints/epoch_{epoch}.pt")

/tmp/ipykernel_1644/3271625938.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Train Epoch 0:   0%|          | 0/957 [00:00<?, ?it/s]/tmp/ipykernel_1644/3271625938.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch 0 Train Loss: 0.8694782581884908


Val Epoch 0:   0%|          | 0/107 [00:00<?, ?it/s]/tmp/ipykernel_1644/3271625938.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Epoch 0 Validation Loss: 0.8522258662731848
Epoch 0 Validation Accuracy: 0.5835294117647059
Saved best model!


Epoch 1 Train Loss: 0.8373675885120778


...

Epoch 12 Validation Loss: 0.9057738037309914
Epoch 12 Validation Accuracy: 0.5329411764705883


Epoch 13 Train Loss: 0.7463154959859271


Epoch 13 Validation Loss: 0.973484973762637
Epoch 13 Validation Accuracy: 0.6788235294117647
Saved best model!


Epoch 14 Train Loss: 0.7512475993628288


Epoch 14 Validation Loss: 0.8377819994342661
Epoch 14 Validation Accuracy: 0.668235294117647


In [ ]:
# -----------------------------
# Save outputs
# -----------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_root = "outputs"
run_dir = os.path.join(output_root, f"llava_{timestamp}")
os.makedirs(run_dir, exist_ok=True)
epochs = range(1, len(val_losses) + 1)

# Save metrics
metrics = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accs": val_accs,
    "epochs": list(epochs),
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY
}
with open(os.path.join(run_dir, "finetune_metrics.json"), "w") as mf:
    import json
    json.dump(metrics, mf, indent=2)

# Loss figure (train & validation)
fig_loss = plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses, marker='o', label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
loss_filename = os.path.join(run_dir, f"loss.png")
plt.savefig(loss_filename)
plt.close(fig_loss)

# Validation accuracy figure
fig_acc = plt.figure(figsize=(6, 4))
plt.plot(epochs, val_accs, marker='o')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.tight_layout()
acc_filename = os.path.join(run_dir, f"val_acc.png")
plt.savefig(acc_filename)
plt.close(fig_acc)


In [ ]:
# Save model
torch.save(model.state_dict(), os.path.join(run_dir, "classifier.pt"))
processor.save_pretrained(os.path.join(run_dir))
model.save_pretrained(os.path.join(run_dir))
torch.save(model, "classifier_full.pt")


print("Training complete, model saved at", run_dir)

Training complete, model saved at outputs/llava_20260317_014817


In [ ]:
model.classifier.save()

In [ ]:
torch.save(model.state_dict(), os.path.join("/content/drive/MyDrive/", "classifier.pt"))


# Evaluation

In [2]:
import os
import glob
import json
from datetime import datetime
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import LlavaForConditionalGeneration, AutoProcessor
from tqdm import tqdm
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

datapath="./data_preprocessed_llava"
output_root="outputs"
batch_size=64

device = "cuda" if torch.cuda.is_available() else "cpu"

data = load_from_disk(datapath)
data.set_format("torch") # ensure tensors

val_dataset = data["validation"]
val_loader = DataLoader(val_dataset, batch_size=batch_size)

Loading dataset from disk:   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
print(f"Loading processor and model to device={device}...")
# Load the base pre-trained model
base_model_name = "llava-hf/llava-1.5-7b-hf"
model = LlavaForConditionalGeneration.from_pretrained(base_model_name).to(device)
processor = AutoProcessor.from_pretrained(base_model_name)

In [7]:
path = "classifier_full.pt"
model = torch.load(path, map_location=device, weights_only=False)

In [ ]:
model

LLaVAClassifier(
  (llava): LlavaForConditionalGeneration(
    (model): LlavaModel(
      (vision_tower): CLIPVisionModel(
        (vision_model): CLIPVisionTransformer(
          (embeddings): CLIPVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
            (position_embedding): Embedding(577, 1024)
          )
          (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (encoder): CLIPEncoder(
            (layers): ModuleList(
              (0-23): 24 x CLIPEncoderLayer(
                (self_attn): CLIPAttention(
                  (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                )
             

In [ ]:
model.llava

LlavaForConditionalGeneration(
  (model): LlavaModel(
    (vision_tower): CLIPVisionModel(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affi

In [17]:
## Clasifier Evaluation

model.eval()

y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Eval", leave=False):
        labels = batch["labels"].to(device)

        # Forward pass
        logits = model(
            pixel_values=batch['pixel_values'].to(device),
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
        )

        # Your classifier outputs shape: [batch_size, 1], flatten to [batch_size]
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
            
        probs = torch.sigmoid(logits)
        # Binary predictions: threshold at 0
        preds = (logits > 0).long()

        # Append predictions, labels, probabilities
        y_pred.extend(preds.cpu().numpy().tolist())
        y_true.extend(labels.cpu().numpy().tolist())
        y_prob.extend(probs.cpu().numpy().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

In [ ]:
## Baseline Evaluation

tokenizer = processor
llava_model = model.llava
llava_model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Baseline LLaVA Eval", leave=False):
        labels = batch["labels"].to("cuda")
        y_true.extend(labels.cpu().numpy().tolist())

        # Generate text predictions
        generated_ids = llava_model.generate(
            input_ids=batch['input_ids'].to("cuda"),
            pixel_values=batch['pixel_values'].to("cuda"),
            attention_mask=batch['attention_mask'].to("cuda"),
            max_new_tokens=20,
            do_sample=False
        )

        # Decode generated text
        texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        # Convert text to binary labels
        for text in texts:
            t = text.lower().strip()
            if "yes" in t or t == "1":
                y_pred.append(1)
            elif "no" in t or t == "0":
                y_pred.append(0)
            else:
                # fallback in case output is unclear
                y_pred.append(0)

# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [12]:
# Metrics
acc = accuracy_score(y_true, y_pred)
prec, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
conf = confusion_matrix(y_true, y_pred).tolist()
roc_auc = None

try:
    # only compute ROC AUC if both classes present in y_true
    if len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_prob))
except Exception:
    roc_auc = None
    
metrics = {
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(recall),
    "f1": float(f1),
    "confusion_matrix": conf,
    "n_samples": int(len(y_true)),
    "roc_auc": roc_auc,
}

# Save metrics
out_json = os.path.join("eval_metrics.json")
with open(out_json, "w") as f:
    json.dump(metrics, f, indent=2)


In [24]:
try:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = None
    if roc_auc is None:
        try:
            auc_val = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc_val = None
    else:
        auc_val = roc_auc

    fig = plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_val:.4f})" if auc_val is not None else "ROC curve")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()
    roc_path = os.path.join(run_dir, "roc_curve.png")
    fig.savefig(roc_path)
    plt.close(fig)
    print(f"Saved ROC curve to {roc_path}")
except Exception as e:
    print(f"Warning: failed to compute/save ROC curve: {e}")

Saved ROC curve to /home/hice1/lcho8/scratch/new-dl/dl-project/llava/outputs/llava_20260502_203023/roc_curve.png


In [ ]:

# Confusion matrix plot
try:
    cm = np.array(conf)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    classes = ["0", "1"]
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("Confusion Matrix")

    # annotate cells with counts
    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(int(cm[i, j]), 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    cm_path = os.path.join("confusion_matrix.png")
    fig.savefig(cm_path)
    plt.close(fig)
    print(f"Saved confusion matrix to {cm_path}")
except Exception as e:
    print(f"Warning: failed to compute/save confusion matrix: {e}")

In [26]:
# generate and save score distribution (histogram) by true label
try:
    fig = plt.figure(figsize=(6, 4))
    pos_scores = y_prob[y_true == 1]
    neg_scores = y_prob[y_true == 0]

    bins = np.linspace(0.0, 1.0, 21)
    # plot both classes on the same axes for comparison
    plt.hist(neg_scores, bins=bins, density=True, alpha=0.6, label="neg (0)", color="C0", edgecolor="black")
    plt.hist(pos_scores, bins=bins, density=True, alpha=0.6, label="pos (1)", color="C1", edgecolor="black")

    plt.xlabel("Predicted probability (positive class)")
    plt.ylabel("Density")
    plt.title("Score Distribution by True Label")
    plt.legend(loc="upper center")
    plt.tight_layout()
    sd_path = os.path.join(run_dir, "score_distribution.png")
    fig.savefig(sd_path)
    plt.close(fig)
    print(f"Saved score distribution to {sd_path}")
except Exception as e:
    print(f"Warning: failed to compute/save score distribution: {e}")

Saved score distribution to /home/hice1/lcho8/scratch/new-dl/dl-project/llava/outputs/llava_20260502_203023/score_distribution.png


In [ ]:
# Print summary
print("\nEvaluation summary:")
print(f"  samples: {metrics['n_samples']}")
print(f"  accuracy: {metrics['accuracy']:.4f}")
print(f"  precision: {metrics['precision']:.4f}")
print(f"  recall: {metrics['recall']:.4f}")
print(f"  f1: {metrics['f1']:.4f}")
print(f"  confusion_matrix: {metrics['confusion_matrix']}")
print(f"Saved metrics to {out_json}")